# 6-DOF Align Experiment Notebook

This notebook sends **both RGB and depth-preview images** to a Gemini VLM with a new
6-DOF prompt (adapted from `web_ai_align_prompt.md`). The VLM reads the JET-colormap
depth image, estimates Z directly, and returns a **full 6-DOF grasp pose** (3D position +
3D orientation) in the camera frame.

After receiving the VLM response we:
1. Parse the 6-DOF JSON
2. Validate the VLM's Z estimate against the ground-truth depth map
3. Visualize grasps on the RGB image (2D)
4. Render an interactive 3D point cloud with gripper wireframes

This extends `align_experiment.ipynb` — the VLM now reasons directly about 3D geometry
rather than returning a 2D point for server-side depth sampling.

In [ ]:
import os, sys, re, json, math, base64, urllib.error, urllib.request
from pathlib import Path
from io import BytesIO
from dataclasses import dataclass
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import colorsys

root = Path('..').resolve()
sys.path.append(str(root))
print('Project root:', root)

from vg_pipeline.io import load_observation_npy
from vg_pipeline.align import (
    sample_depth_median,
    deproject_pixel,
    grasp_scale_anchor,
)
from vg_pipeline.roi import _load_vlm_json, _scale_norm_xy_to_rgb
from grasp_server.align_grasp import (
    _pose_from_point_and_angle,
    _DEFAULT_WIDTH_M,
)
from grasp_server.grasp_selection import _rotation_to_quaternion_xyzw
from vg_pipeline.geometry import backproject_depth_with_mask
from vg_pipeline.grasp_results import GRIPPER_DEPTH_METERS, FINGER_LENGTH_METERS

print(
    'Imported project helpers: load_observation_npy, sample_depth_median, deproject_pixel,\n'
    'grasp_scale_anchor, _load_vlm_json, _scale_norm_xy_to_rgb, _pose_from_point_and_angle,\n'
    '_rotation_to_quaternion_xyzw, backproject_depth_with_mask'
)

## Part 2: API Key Setup

Load the Gemini/Google API key from environment variables or shell rc files.

In [ ]:
# Gemini/VLM API key: prefer the environment, else read it from the zsh rc files (macOS).
def _api_key_from_zsh_rc():
    for rc in ('.zshrc', '.zshenv', '.zprofile'):
        path = Path.home() / rc
        if not path.exists():
            continue
        m = re.search(r'export\s+(GEMINI_API_KEY|GOOGLE_API_KEY)=[\"\']?([^\"\'\n]+)', path.read_text())
        if m:
            return m.group(1), m.group(2)
    return None, None

api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY')
api_key_name = 'GEMINI_API_KEY' if os.environ.get('GEMINI_API_KEY') else 'GOOGLE_API_KEY' if api_key else None
if api_key is None:
    api_key_name, api_key = _api_key_from_zsh_rc()
    if api_key:
        os.environ[api_key_name] = api_key
        print(f'Loaded {api_key_name} from shell rc file.')

if api_key is None:
    raise RuntimeError('Set GEMINI_API_KEY or GOOGLE_API_KEY (e.g. export it in ~/.zshrc).')

print('Using API key from env var:', api_key_name)

## Part 3: Select Capture & Load Data

Choose a capture folder from `rgbd_data/captures/`. Load the RGB image (converted from BGR),
depth map (float32 meters), camera intrinsics, and the depth-preview image (JET colormap)
that the VLM will use for Z estimation.

In [ ]:
# Pick a capture folder — change to any folder in rgbd_data/captures/
capture_name = '20260417_115700'
capture_dir = root / 'rgbd_data' / 'captures' / capture_name

camera_data = load_observation_npy(capture_dir / 'camera_data.npy')

# Extract data
rgb_bgr = camera_data['rgb'].astype(np.uint8)          # BGR uint8 (720, 1280, 3)
depth_map = camera_data['depth'].astype(np.float32)    # float32 meters (720, 1280)
K = camera_data['K'].astype(np.float64)                # 3x3 intrinsic matrix

# Convert BGR -> RGB PIL image (for display & API)
rgb_np = rgb_bgr[:, :, ::-1]  # BGR -> RGB
rgb_pil = Image.fromarray(rgb_np, mode='RGB')

# Load depth preview (JET colormap, pre-saved alongside captures, aligned to RGB)
depth_pil = Image.open(capture_dir / 'depth_preview.jpg').convert('RGB')

print(f"Capture dir : {capture_dir}")
print(f"RGB size    : {rgb_pil.size}")
print(f"Depth size  : {depth_pil.size}")
print(f"Depth range : [{np.nanmin(depth_map):.3f}, {np.nanmax(depth_map):.3f}] m")
print(f"Camera K:\n{K}")

## Part 4: Build 6-DOF Align Prompt

This prompt adapts `web_ai_align_prompt.md` for programmatic API use. It:
- Uses the **actual camera intrinsics** from the capture (not the approximate 640 values)
- Includes a **scale reference** computed from depth + intrinsics via `grasp_scale_anchor`
- References **both images** (RGB first, then depth-preview in JET colormap)
- Asks the VLM to output full 6-DOF JSON: `position_3d`, `closing_direction_3d`, `approach_direction_3d`, `pixel_uv`, etc.

In [ ]:
# =============================================================================
# 6-DOF Align Prompt Template — adapted from web_ai_align_prompt.md
# Uses actual camera K values and scale anchor computed from the depth map.
# =============================================================================

ALIGN_6DOF_PROMPT_TEMPLATE = (
    "You are an expert in Embodied AI and robot vision. Your task is to output a "
    "**full 6-DOF grasp pose** (3D position + 3D orientation) in the camera coordinate "
    "frame. This is NOT a 2D bounding box task — you must reason about the 3D geometry "
    "of the scene and output coordinates in **meters and 3D vectors**.\n\n"
    "You receive two images of the same scene (1280x720, pixel-aligned):\n"
    "- **Image 1 (RGB)**: standard color image showing the scene appearance.\n"
    "- **Image 2 (Depth, JET colormap)**: a false-color visualization of the depth map. "
    "Warmer colors (red/yellow) = closer to the camera; cooler colors (blue/purple) = "
    "farther away. Pixel (u, v) in both images corresponds to the same 3D point.\n\n"
    "## Camera Intrinsics & Coordinate System\n\n"
    "The images come from an **Intel RealSense D455** RGB-D camera. The depth is already "
    "aligned to the RGB frame. **All coordinates below are in the camera frame.**\n\n"
    "### Actual Intrinsic Matrix K\n\n"
    "```\n"
    "K = [[{fx:.4f},  0,      {cx:.4f}],\n"
    "     [ 0,      {fy:.4f}, {cy:.4f}],\n"
    "     [ 0,      0,        1     ]]\n"
    "```\n\n"
    "### Pinhole Projection (the math you MUST use)\n\n"
    "**2D pixel -> 3D position (deprojection):**\n"
    "```\n"
    "Given: pixel (u, v) and depth Z (meters)\n"
    "Compute:\n"
    "  X = (u - cx) * Z / fx\n"
    "  Y = (v - cy) * Z / fy\n"
    "  Z = Z\n"
    "```\n\n"
    "**3D position -> 2D pixel (projection):**\n"
    "```\n"
    "Given: 3D point (X, Y, Z) in camera frame\n"
    "Compute:\n"
    "  u = fx * (X / Z) + cx\n"
    "  v = fy * (Y / Z) + cy\n"
    "```\n\n"
    "### Camera Coordinate Frame\n\n"
    "```\n"
    "       +Y (image down)\n"
    "       |\n"
    "       |\n"
    "       +-------> +X (image right)\n"
    "      /\n"
    "     /\n"
    "   +Z (into the scene, perpendicular to the image plane)\n"
    "```\n"
    "- **Origin**: camera optical center\n"
    "- **+X**: points to the right in the image\n"
    "- **+Y**: points downward in the image\n"
    "- **+Z**: points forward, perpendicularly into the scene\n\n"
    "The table/support surface is roughly parallel to the XZ plane. The camera is "
    "mounted above and looks slightly downward.\n\n"
    "### How to Estimate Depth from the JET Visualization\n\n"
    "The depth image (Image 2) uses the JET colormap. To estimate depth Z at a pixel:\n\n"
    "1. Read the color at your target pixel in the depth image\n"
    "2. Map it to an approximate depth:\n"
    "   - **Dark red / maroon**: ~0.3-0.4 m (very close)\n"
    "   - **Red / orange**: ~0.4-0.55 m\n"
    "   - **Yellow**: ~0.55-0.7 m\n"
    "   - **Green**: ~0.7-1.0 m\n"
    "   - **Cyan**: ~1.0-1.3 m\n"
    "   - **Blue**: ~1.3-1.8 m\n"
    "   - **Dark blue / purple**: ~1.8-2.5+ m (far)\n"
    "3. Interpolate between these bands — neighboring pixels give clues about 3D "
    "surface orientation.\n"
    "4. Be conservative: if the depth color changes rapidly around your point "
    "(edges, occlusions), the depth is unreliable — pick a different point.\n\n"
    "## Robot Gripper Specification\n\n"
    "The robot uses a **parallel-jaw gripper**:\n\n"
    "| Parameter | Value |\n"
    "|---|---|\n"
    "| Type | Parallel-jaw, two-finger |\n"
    "| Max jaw opening | **0.08 m (80 mm)** |\n"
    "| Min useful opening | **0.02 m (20 mm)** |\n"
    "| Approach direction | **Fixed as camera +Z** = [0, 0, 1] |\n"
    "| Closing direction rotation plane | **XY plane only** (image plane) |\n"
    "| Gripper angle convention | **0deg = jaws close along +X (image right); "
    "positive = rotate toward +Y (image down)** |\n\n"
    "### Full 6-DOF Pose Construction\n\n"
    "The 6-DOF grasp pose consists of:\n\n"
    "1. **Position** (3 DOF): `[X, Y, Z]` in meters, camera frame — the 3D point "
    "where the gripper center aligns with the object.\n\n"
    "2. **Orientation** (3 DOF), defined by two orthogonal axes:\n"
    "   - **Approach axis**: fixed as `[0, 0, 1]` — the gripper always approaches "
    "perpendicular to the image (along camera +Z).\n"
    "   - **Closing axis**: `[cos(theta), sin(theta), 0]` — the direction the two "
    "jaws close, rotating within the image plane. This is the ONLY orientation degree "
    "of freedom you control, parameterized by `gripper_angle_deg` = theta.\n"
    "   - **Lateral axis**: automatically = approach x closing = "
    "`[-sin(theta), cos(theta), 0]`\n\n"
    "In other words, the full 6-DOF pose is:\n\n"
    "```\n"
    "Position:       [X, Y, Z]           <- you compute from pixel + estimated depth\n"
    "Closing dir:    [cos(theta), sin(theta), 0]  <- you choose theta = gripper_angle_deg\n"
    "Approach dir:   [0, 0, 1]            <- fixed\n"
    "```\n\n"
    "{scale_reference_block}"
    "## Your Task\n\n"
    "Given the RGB and depth images, output the **full 6-DOF grasp pose** for "
    "`{task_spec}`. Propose `{num_candidates}` diverse candidates ranked from best "
    "to worst.\n\n"
    "### Reasoning Steps\n\n"
    "1. **Identify the target**: find `{task_spec}` in the RGB image. Cross-reference "
    "with the depth image to understand its full 3D shape — where it sits in 3D space, "
    "how thick it is, where it separates from the background.\n\n"
    "2. **Estimate 3D position for each candidate**: for each grasp point you consider:\n"
    "   - Locate the pixel (u, v) in the RGB image\n"
    "   - Look up the SAME pixel in the depth image — what color is it?\n"
    "   - Estimate Z (meters) from the depth color using the JET colormap\n"
    "   - Compute **X = (u - cx) * Z / fx** and **Y = (v - cy) * Z / fy** using "
    "the actual K values above\n"
    "   - This gives you the **3D position [X, Y, Z]** in camera frame\n\n"
    "3. **Diversity planning**: mentally divide the target into distinct spatial zones "
    "(upper/middle/lower, left/right, near/far based on depth). Assign each candidate "
    "to a different zone. At least one candidate must have a gripper_angle_deg >= 30deg "
    "different from the others.\n\n"
    "4. **Choose gripper angle (theta)**: for each candidate, determine "
    "`gripper_angle_deg` such that:\n"
    "   - The closing direction `[cos(theta), sin(theta), 0]` crosses the object's "
    "**narrow** dimension at that point\n"
    "   - The object width along the closing direction fits in [20 mm, 80 mm]\n"
    "   - Estimate the pixel width along the closing direction, then convert to "
    "meters: width_m = width_px * Z / fx\n"
    "   - Reject orientations where width_m > 0.08 m (won't fit) or "
    "width_m < 0.02 m (too thin, unstable)\n\n"
    "5. **Table clearance**: the table is visible in the depth image as a large flat "
    "region. The grasp Y should be above the object-table contact line. If the depth "
    "at your point is the same as the table depth, you're on the table, not the object.\n\n"
    "6. **Self-check**: verify:\n"
    "   - No two candidates' 3D positions are within ~3 cm of each other\n"
    "   - Every position has table clearance\n"
    "   - Every closing-direction width fits [20 mm, 80 mm]\n"
    "   - The pixel (u, v) maps to solid object surface in both images\n\n"
    "### What to Avoid\n"
    "- Handles, spouts, edges, tips, weak joints, high-curvature regions\n"
    "- Transparent, reflective, or depth-less regions (if the depth image shows a "
    "dark hole or speckled noise, avoid it)\n"
    "- The contact line between object and table (visible in depth as a sharp transition)\n"
    "- Orientations where the closing span clearly exceeds 80 mm\n\n"
    "## Output Format\n\n"
    "Output ONLY a JSON object (no markdown fences, no extra text before or after):\n\n"
    "{{\n"
    '  "target": "{task_spec}",\n'
    '  "image_size": [{h}, {w}],\n'
    '  "camera_intrinsics_used": {{"fx": {fx:.4f}, "fy": {fy:.4f}, "cx": {cx:.4f}, "cy": {cy:.4f}}},\n'
    '  "candidates": [\n'
    '    {{\n'
    '      "rank": 1,\n'
    '      "position_3d": [X, Y, Z],\n'
    '      "gripper_angle_deg": 45.0,\n'
    '      "closing_direction_3d": [cx, cy, cz],\n'
    '      "approach_direction_3d": [0.0, 0.0, 1.0],\n'
    '      "pixel_uv": [u, v],\n'
    '      "align_point_norm": [y_0_1000, x_0_1000],\n'
    '      "estimated_depth_m": 0.55,\n'
    '      "estimated_object_width_along_close_m": 0.045,\n'
    '      "reasoning": "Zone: <name>. Depth color: <color> -> Z~<value>m. '
    "Position computed: X=(u-cx)*Z/fx=..., Y=(v-cy)*Z/fy=..., Z=... "
    "Why this angle, how this candidate differs from others, why width fits. "
    'Avoid curly braces in reasoning text."\n'
    '    }}\n'
    '  ]\n'
    '}}\n\n'
    "### Field Descriptions\n\n"
    "| Field | Type | Description |\n"
    "|---|---|---|\n"
    "| `position_3d` | `[X, Y, Z]` | **6-DOF position** in camera frame, meters. "
    "Computed via deprojection from pixel + estimated depth. |\n"
    "| `gripper_angle_deg` | number | In-image closing rotation. 0deg = horizontal "
    "(+X), 90deg = vertical (+Y). |\n"
    "| `closing_direction_3d` | `[cx, cy, cz]` | **6-DOF orientation**: unit vector "
    "of jaw closing direction. Must be `[cos(theta), sin(theta), 0]` with cz=0. |\n"
    "| `approach_direction_3d` | `[0, 0, 1]` | **6-DOF orientation**: approach "
    "direction. Always camera +Z. |\n"
    "| `pixel_uv` | `[u, v]` | The exact pixel coordinates (not normalized) where you "
    "placed the grasp. Used for server-side refinement with precise depth. |\n"
    "| `align_point_norm` | `[y, x]` | Same point, normalized to 0-1000 range. "
    "0 = top/left, 1000 = bottom/right. Fallback for the existing server pipeline. |\n"
    "| `estimated_depth_m` | number | Your Z estimate in meters from reading the "
    "depth JET colormap. |\n"
    "| `estimated_object_width_along_close_m` | number | Your estimate of the object's "
    "width along the closing direction, in meters. Must be in [0.02, 0.08]. |\n"
    "| `reasoning` | string | Must include: zone name, depth color observed, Z estimate, "
    "deprojection calculation, how this differs from other candidates, and width "
    "feasibility confirmation. Do NOT use curly braces in this field. |\n\n"
    "### The 6-DOF Pose in Summary\n\n"
    "Each candidate fully defines a grasp in 3D space:\n\n"
    "```\n"
    "Frame: CAMERA (not world)\n\n"
    "Position:    position_3d = [X, Y, Z]           <- meters, from pixel + estimated depth\n"
    "Closing:     closing_direction_3d = [cos(theta), sin(theta), 0]\n"
    "Lateral:     approach x closing = [-sin(theta), cos(theta), 0]   (computed automatically)\n"
    "Approach:    approach_direction_3d = [0, 0, 1]  <- fixed, perpendicular to image\n\n"
    "This is a complete right-handed 6-DOF grasp frame.\n"
    "```\n\n"
    "**You are the one computing the 3D position.** Do NOT just output 2D pixel "
    "coordinates. Use the depth image to estimate Z, then apply the pinhole "
    "deprojection formula yourself. Show your work in the `reasoning` field.\n"
    "- `closing_direction_3d` cz must be exactly 0 — the closing happens in the "
    "image plane only.\n"
    "- `pixel_uv` must land on solid object surface.\n"
    "- If you see the depth image has large black/noisy regions (missing depth, "
    "common with reflective or very dark surfaces), do NOT place grasp points there.\n"
)


def build_6dof_align_prompt(
    task_spec: str,
    w: int,
    h: int,
    num_candidates: int,
    *,
    K: np.ndarray,
    max_open_px: float,
    min_open_px: float,
    ref_depth_m: float,
    gripper_max_open_m: float = 0.08,
    gripper_min_open_m: float = 0.02,
) -> str:
    """Build a 6-DOF align prompt with actual camera intrinsics and scale reference.

    Parameters
    ----------
    task_spec : str
        Target object description (e.g. "the rail").
    w, h : int
        Image width and height in pixels.
    num_candidates : int
        Number of diverse candidates to request.
    K : np.ndarray
        3x3 camera intrinsic matrix.
    max_open_px, min_open_px : float
        Jaw opening limits in pixels at the reference depth (from
        :func:`vg_pipeline.align.grasp_scale_anchor`).
    ref_depth_m : float
        Representative scene depth in meters.
    gripper_max_open_m, gripper_min_open_m : float
        Metric jaw opening limits (default 0.08 / 0.02).
    """
    K = np.asarray(K, dtype=np.float64)
    fx, fy = float(K[0, 0]), float(K[1, 1])
    cx, cy = float(K[0, 2]), float(K[1, 2])

    scale_reference_block = (
        f"### Scale Reference\n\n"
        f"At depth Z meters, one pixel = **Z / fx** meters. "
        f"Gripper opening converted to pixel span:\n\n"
        f"> pixels_needed = gripper_opening_mm / 1000 * fx / Z\n\n"
        f"At the scene's representative working depth (~{ref_depth_m:.2f} m) "
        f"in this {w}x{h} image:\n"
        f"- The jaw's maximum opening "
        f"{gripper_max_open_m * 1000:.0f} mm spans about **{max_open_px:.0f} px**\n"
        f"- The jaw's minimum opening "
        f"{gripper_min_open_m * 1000:.0f} mm spans about **{min_open_px:.0f} px**\n\n"
        f"Use these pixel sizes to judge whether the object's width along the closing "
        f"direction fits the jaw.\n\n"
    )

    return ALIGN_6DOF_PROMPT_TEMPLATE.format(
        task_spec=task_spec.strip(),
        w=w,
        h=h,
        num_candidates=num_candidates,
        fx=fx,
        fy=fy,
        cx=cx,
        cy=cy,
        scale_reference_block=scale_reference_block,
    )

In [ ]:
# ---- Compute scale anchor & build the prompt ----

task_spec = 'the rail'
num_candidates = 2

z_ref, max_open_px, min_open_px = grasp_scale_anchor(depth_map, K)
print(f'Scale anchor: z_ref={z_ref:.3f} m  max_open={max_open_px:.0f} px  min_open={min_open_px:.0f} px')

prompt_6dof = build_6dof_align_prompt(
    task_spec,
    w=rgb_pil.width,
    h=rgb_pil.height,
    num_candidates=num_candidates,
    K=K,
    max_open_px=max_open_px,
    min_open_px=min_open_px,
    ref_depth_m=z_ref,
)

print(f'Using task spec : {task_spec}')
print(f'Candidates      : {num_candidates}')
print(f'Prompt length   : {len(prompt_6dof)} chars')
print('--- Prompt preview (first 600 chars) ---')
print(prompt_6dof[:600])
print('...')
print('--- Prompt preview (last 400 chars) ---')
print(prompt_6dof[-400:])

## Part 5: Run VLM Inference (Two Images)

Send both the **RGB image** and **depth-preview image** to the Gemini API. The API call
is constructed inline because `run_vg_inference` / `run_gemini_vg` validate exactly one
image — we bypass them and construct the REST call directly with two `inline_data` parts.

In [ ]:
# ---- Inline Gemini API call with TWO images ----
# Same pattern as vg_pipeline/providers.py::run_gemini_vg but without the
# single-image restriction.

def _pil_to_b64(image: Image.Image, mime_type: str = 'image/png') -> str:
    """Encode a PIL image to a base64 data string (same as providers._pil_to_gemini_base64)."""
    buf = BytesIO()
    im = image
    if im.mode not in ('RGB', 'RGBA'):
        im = im.convert('RGB')
    if mime_type == 'image/jpeg':
        im.save(buf, format='JPEG', quality=92)
    else:
        im.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('ascii')


# Model: change to any Gemini model that supports vision
model_name = 'gemini-robotics-er-1.6-preview'

parts: list[dict[str, Any]] = []

# Image 1: RGB (PNG for lossless)
parts.append({
    'inline_data': {
        'mime_type': 'image/png',
        'data': _pil_to_b64(rgb_pil, 'image/png'),
    }
})

# Image 2: Depth preview (JPEG — it is already a lossy JPG)
parts.append({
    'inline_data': {
        'mime_type': 'image/jpeg',
        'data': _pil_to_b64(depth_pil, 'image/jpeg'),
    }
})

# Prompt text
parts.append({'text': prompt_6dof})

payload = {
    'contents': [{'parts': parts}],
    'generationConfig': {'temperature': 0},
}

endpoint = (
    f'https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent'
    f'?key={api_key}'
)

print(f'Calling Gemini API: {model_name} ...')
print(f'Payload: {len(payload["contents"][0]["parts"])} parts '
      f'(2 images + 1 text, ~{len(prompt_6dof)} chars prompt)')

req = urllib.request.Request(
    endpoint,
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)

try:
    with urllib.request.urlopen(req) as resp:
        body = json.loads(resp.read().decode('utf-8'))
except urllib.error.HTTPError as exc:
    detail = exc.read().decode('utf-8', errors='ignore')
    raise RuntimeError(f'Gemini API request failed: {detail}') from exc
except urllib.error.URLError as exc:
    raise RuntimeError(f'Gemini API connection failed: {exc}') from exc

# Extract text from response
try:
    resp_parts = body['candidates'][0]['content']['parts']
    raw_model_text = '\n'.join(
        part.get('text', '') for part in resp_parts if 'text' in part
    ).strip()
except (KeyError, IndexError, TypeError) as exc:
    raise RuntimeError(f'Unexpected Gemini response: {body}') from exc

print(f'\\nRaw VLM response ({len(raw_model_text)} chars):')
print(raw_model_text[:3000])

## Part 6: Parse 6-DOF Results

Parse the VLM JSON into `Align6DoFResult` dataclass instances. The parser handles:
- **New 6-DOF format**: `position_3d`, `closing_direction_3d`, `pixel_uv` (primary)
- **Fallbacks**: `align_point_norm` → pixel, `gripper_angle_deg` → closing direction

In [ ]:
@dataclass(frozen=True)
class Align6DoFResult:
    """Parsed 6-DOF alignment proposal from the VLM.

    All fields except ``rank`` and ``angle_deg`` can be ``None`` when the VLM
    returns a minimal (2D-only) response — the fallback logic in the parser
    populates as much as possible.
    """
    rank: int
    position_3d: np.ndarray | None          # [X, Y, Z] in camera frame, meters
    closing_direction_3d: np.ndarray | None  # unit vector, in image plane (cz=0)
    approach_direction_3d: np.ndarray | None # fixed [0, 0, 1]
    pixel_uv: tuple[int, int] | None        # raw pixel (u, v) from VLM
    align_point_yx: tuple[int, int] | None  # full-res pixel from align_point_norm
    angle_deg: float                        # in-image gripper angle
    estimated_depth_m: float | None         # VLM's Z estimate from JET colormap
    estimated_width_m: float | None         # VLM's object-width estimate
    reasoning: str | None


def _try_parse_float_list(raw, expected_len=None):
    """Parse a list of floats from raw JSON value. Returns None on failure."""
    if raw is None or not isinstance(raw, (list, tuple)):
        return None
    try:
        vals = [float(v) for v in raw]
    except (ValueError, TypeError):
        return None
    if expected_len is not None and len(vals) != expected_len:
        return None
    return vals


def parse_align_6dof_results(
    raw_text: str,
    *,
    canvas_h: int,
    canvas_w: int,
    rgb_h: int,
) -> list[Align6DoFResult]:
    """Parse VLM JSON with ``candidates`` array into :class:`Align6DoFResult` list.

    Handles the new 6-DOF format (``position_3d``, ``closing_direction_3d``,
    ``pixel_uv``, etc.) with fallbacks to the existing 2D format (``align_point``,
    ``gripper_angle_deg``).
    """
    data = _load_vlm_json(raw_text)
    candidates = data.get('candidates')
    if not isinstance(candidates, list) or len(candidates) == 0:
        raise ValueError('VLM returned no candidates:\n' + str(data))

    results: list[Align6DoFResult] = []
    for item in candidates:
        # --- rank ---
        rank_raw = item.get('rank', 0)
        rank = int(rank_raw) if isinstance(rank_raw, (int, float)) else 0

        # --- pixel_uv (primary) + align_point_norm (fallback) ---
        pixel_uv: tuple[int, int] | None = None
        pu = _try_parse_float_list(item.get('pixel_uv'), expected_len=2)
        if pu is not None:
            pixel_uv = (int(round(pu[0])), int(round(pu[1])))

        # Fallback: align_point or align_point_norm
        align_point_yx: tuple[int, int] | None = None
        ap = item.get('align_point_norm') or item.get('align_point')
        if ap is not None:
            ap_vals = _try_parse_float_list(ap, expected_len=2)
            if ap_vals is not None:
                align_point_yx = _scale_norm_xy_to_rgb(
                    ap_vals[0], ap_vals[1],
                    canvas_h=canvas_h, canvas_w=canvas_w, rgb_h=rgb_h,
                )
                if pixel_uv is None:
                    pixel_uv = (align_point_yx[1], align_point_yx[0])  # (u, v) from (y, x)

        # --- closing_direction_3d (primary) + gripper_angle_deg (fallback) ---
        closing_direction_3d: np.ndarray | None = None
        cd = _try_parse_float_list(item.get('closing_direction_3d'), expected_len=3)
        if cd is not None:
            arr = np.array(cd, dtype=np.float64)
            n = np.linalg.norm(arr)
            if n > 1e-9:
                arr = arr / n
            # Clamp tiny cz to 0
            if abs(arr[2]) < 1e-6:
                arr[2] = 0.0
            closing_direction_3d = arr

        angle_deg: float = 0.0
        angle_raw = item.get('gripper_angle_deg')
        if isinstance(angle_raw, (int, float)):
            angle_deg = float(angle_raw)

        # If closing direction is missing but angle is present, compute it
        if closing_direction_3d is None:
            th = math.radians(angle_deg)
            closing_direction_3d = np.array(
                [math.cos(th), math.sin(th), 0.0], dtype=np.float64
            )
        else:
            # Override angle_deg from the closing direction for consistency
            angle_deg = math.degrees(math.atan2(
                closing_direction_3d[1], closing_direction_3d[0]
            ))

        # --- approach_direction_3d (always [0, 0, 1]) ---
        approach_direction_3d: np.ndarray | None = None
        ad = _try_parse_float_list(item.get('approach_direction_3d'), expected_len=3)
        if ad is not None:
            approach_direction_3d = np.array(ad, dtype=np.float64)
        else:
            approach_direction_3d = np.array([0.0, 0.0, 1.0], dtype=np.float64)

        # --- position_3d ---
        position_3d: np.ndarray | None = None
        p3 = _try_parse_float_list(item.get('position_3d'), expected_len=3)
        if p3 is not None:
            position_3d = np.array(p3, dtype=np.float64)

        # --- estimated_depth_m ---
        estimated_depth_m: float | None = None
        ed = item.get('estimated_depth_m')
        if isinstance(ed, (int, float)):
            estimated_depth_m = float(ed)

        # --- estimated_width_m ---
        estimated_width_m: float | None = None
        ew = item.get('estimated_object_width_along_close_m')
        if isinstance(ew, (int, float)) and float(ew) > 0:
            estimated_width_m = float(ew)
        else:
            # Fallback: existing format uses width_mm
            wm = item.get('width_mm')
            if isinstance(wm, (int, float)) and float(wm) > 0:
                estimated_width_m = float(wm) / 1000.0

        # --- reasoning ---
        reasoning: str | None = item.get('reasoning')
        if not isinstance(reasoning, str):
            reasoning = None

        results.append(Align6DoFResult(
            rank=rank,
            position_3d=position_3d,
            closing_direction_3d=closing_direction_3d,
            approach_direction_3d=approach_direction_3d,
            pixel_uv=pixel_uv,
            align_point_yx=align_point_yx,
            angle_deg=angle_deg,
            estimated_depth_m=estimated_depth_m,
            estimated_width_m=estimated_width_m,
            reasoning=reasoning,
        ))

    return results


# ---- Run parser ----
parsed_6dof = parse_align_6dof_results(
    raw_model_text,
    canvas_h=rgb_pil.height,
    canvas_w=rgb_pil.width,
    rgb_h=rgb_pil.height,
)

if not parsed_6dof:
    raise RuntimeError('No candidates parsed from VLM response.')

print(f'Parsed {len(parsed_6dof)} candidate(s):\n')
for r in parsed_6dof:
    print(f"  #{r.rank}: pixel_uv={r.pixel_uv}  angle={r.angle_deg:.1f}°")
    if r.position_3d is not None:
        print(f"          VLM pos_3d  = [{r.position_3d[0]:.4f}, {r.position_3d[1]:.4f}, {r.position_3d[2]:.4f}] m")
    if r.closing_direction_3d is not None:
        print(f"          VLM closing = [{r.closing_direction_3d[0]:.4f}, {r.closing_direction_3d[1]:.4f}, {r.closing_direction_3d[2]:.4f}]")
    if r.estimated_depth_m is not None:
        print(f"          VLM est Z   = {r.estimated_depth_m:.3f} m")
    if r.estimated_width_m is not None:
        print(f"          VLM width   = {r.estimated_width_m*1000:.0f} mm")
    if r.reasoning:
        print(f"          reasoning   = {r.reasoning[:120]}...")
    print()

## Part 7: Validate Estimated Depth

For each candidate, sample the **ground-truth depth** at `pixel_uv` using a 5x5 median
window and compare against the VLM's `estimated_depth_m` (from reading the JET colormap).

In [ ]:
print(f"{'Rank':<6} {'pixel_uv':<14} {'VLM est Z':<10} {'Actual Z':<14} {'Error cm':<10}")
print("-" * 56)
for r in parsed_6dof:
    if r.pixel_uv is None:
        print(f"{r.rank:<6} {'(missing)':<14} {'N/A':<10} {'N/A':<14} {'N/A':<10}")
        continue
    u, v = r.pixel_uv
    try:
        z_actual = sample_depth_median(depth_map, v, u, window=5)
        z_est = r.estimated_depth_m if r.estimated_depth_m is not None else float('nan')
        error_cm = (z_est - z_actual) * 100
        print(f"{r.rank:<6} ({u},{v}){'':>6} {z_est:<10.3f} {z_actual:<14.4f} {error_cm:+<11.1f}")
    except ValueError:
        print(f"{r.rank:<6} ({u},{v}){'':>6} {z_est:<10.3f} {'(no depth)':<14} {'N/A':<10}")

print()
print("Note: 'VLM est Z' is from the VLM reading the JET colormap (approximate).")
print("'Actual Z' is sampled from the raw float32 depth map (millimeter-accurate).")
print("Large errors mean the VLM misread the depth color.")

## Part 8: 2D Grasp Visualization

Overlay the 6-DOF grasp candidates on the RGB image. For each candidate:
- **Center dot** at `pixel_uv` (where the VLM intended to grasp)
- **Closing-direction arrow** (3D → projected with actual K)
- **Approach-direction arrow** (+Z, desaturated colour)
- **Diagnostic cross** if `position_3d` projects far from `pixel_uv`
- **Rank label**, depth estimate, and 3D position text

The drawing helpers are copied from `align_experiment.ipynb` Part 8.

In [ ]:
# =============================================================================
# Drawing helpers (copied from align_experiment.ipynb Part 8)
# =============================================================================

def _rank_color(rank_index, total):
    """HSV sweep: blue (rank 1) -> green -> orange/red (last rank)."""
    hue = 0.62 - (0.55 * (rank_index / max(total, 1)))
    r, g, b = colorsys.hsv_to_rgb(hue % 1.0, 0.85, 1.0)
    return int(255 * r), int(255 * g), int(255 * b)


def _project(xyz, K_img):
    """Pinhole projection: 3D camera-frame point -> 2D pixel (u, v)."""
    x, y, z = float(xyz[0]), float(xyz[1]), float(xyz[2])
    if z <= 0.0:
        return None
    u = K_img[0, 0] * x / z + K_img[0, 2]
    v = K_img[1, 1] * y / z + K_img[1, 2]
    return float(u), float(v)


def _draw_arrow(draw, start_xy, end_xy, color, line_width=3):
    """Draw a line with a triangular arrowhead."""
    draw.line([start_xy, end_xy], fill=color, width=line_width)
    sx, sy = start_xy
    ex, ey = end_xy
    dx, dy = ex - sx, ey - sy
    length = math.hypot(dx, dy)
    if length < 1e-6:
        return
    ux, uy = dx / length, dy / length       # unit vector along the line
    lx, ly = -uy, ux                         # perpendicular (left)
    head_len = max(10, min(18, length * 0.22))
    head_w = head_len * 0.45
    tip = (ex, ey)
    base = (ex - head_len * ux, ey - head_len * uy)
    left_wing = (base[0] + head_w * lx, base[1] + head_w * ly)
    right_wing = (base[0] - head_w * lx, base[1] - head_w * ly)
    draw.polygon([tip, left_wing, right_wing], fill=color)


def _try_load_font(size):
    """Load a reasonable font on macOS / Linux, falling back to PIL default."""
    import platform
    paths = []
    if platform.system() == 'Darwin':
        paths = [
            '/System/Library/Fonts/Helvetica.ttc',
            '/System/Library/Fonts/Supplemental/Arial.ttf',
            '/Library/Fonts/Arial.ttf',
        ]
    else:
        paths = [
            '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
            '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
        ]
    for p in paths:
        try:
            return ImageFont.truetype(p, size)
        except (IOError, OSError):
            continue
    return ImageFont.load_default()


def draw_web_ai_grasps(rgb_pil, candidates, K_actual):
    """Overlay 6-DOF grasp candidates onto an RGB image.

    For each candidate this draws:
      - Center dot at ``pixel_uv``
      - Closing-direction arrow (3D -> projected with actual K)
      - Approach-direction arrow (+Z, desaturated colour)
      - Diagnostic cross if ``position_3d`` projects far from ``pixel_uv``
      - Rank label, depth estimate, and 3D position text
    """
    img = rgb_pil.copy()
    draw = ImageDraw.Draw(img)
    total = len(candidates)
    dot_r = max(8, min(img.width, img.height) // 80)
    font = _try_load_font(dot_r + 2)
    font_sm = _try_load_font(max(8, dot_r - 2))

    for c in candidates:
        rank = c.get('rank', 0)
        color = _rank_color(rank - 1, total)
        light = tuple(min(255, ch + 80) for ch in color)

        # ---- centre dot ----
        pixel_uv = c.get('pixel_uv')
        if pixel_uv is None:
            print(f"  [WARN] Candidate #{rank}: missing pixel_uv, skipping")
            continue
        uc, vc = int(pixel_uv[0]), int(pixel_uv[1])
        r = dot_r
        draw.ellipse([uc - r, vc - r, uc + r, vc + r],
                     fill=color, outline='black', width=2)

        pos_3d = c.get('position_3d')
        close_dir = c.get('closing_direction_3d')
        angle_deg = c.get('gripper_angle_deg')

        # Normalise closing direction (fall back to angle_deg)
        if close_dir is not None:
            close_dir = np.array(close_dir, dtype=np.float64)
            n = np.linalg.norm(close_dir)
            if n > 1e-9:
                close_dir = (close_dir / n).tolist()
            if abs(close_dir[2]) < 1e-6:
                close_dir[2] = 0.0
        elif angle_deg is not None:
            th = math.radians(angle_deg)
            close_dir = [math.cos(th), math.sin(th), 0.0]

        # ---- closing arrow (projected from 3D) ----
        if pos_3d and close_dir:
            start_uv = _project(pos_3d, K_actual)
            end_3d = [pos_3d[0] + 0.05 * close_dir[0],
                      pos_3d[1] + 0.05 * close_dir[1],
                      pos_3d[2] + 0.05 * close_dir[2]]
            end_uv = _project(end_3d, K_actual)
            if start_uv and end_uv:
                _draw_arrow(draw, start_uv, end_uv, color, line_width=3)

        # ---- approach arrow (+Z, into the scene) ----
        if pos_3d:
            start_uv = _project(pos_3d, K_actual)
            app_end_3d = [pos_3d[0], pos_3d[1], pos_3d[2] + 0.08]
            app_end_uv = _project(app_end_3d, K_actual)
            if start_uv and app_end_uv:
                _draw_arrow(draw, start_uv, app_end_uv, light, line_width=2)

        # ---- diagnostic cross: where position_3d projects vs pixel_uv ----
        if pos_3d:
            proj = _project(pos_3d, K_actual)
            if proj:
                pu, pv = int(proj[0]), int(proj[1])
                dist = math.hypot(pu - uc, pv - vc)
                if dist > 5:
                    cs = max(3, r // 2)
                    draw.line([pu - cs, pv, pu + cs, pv], fill=color, width=2)
                    draw.line([pu, pv - cs, pu, pv + cs], fill=color, width=2)

        # ---- text labels ----
        lx = uc + r + 4
        ly = vc - r

        rank_str = f"#{rank}"
        draw.text((lx + 1, ly + 1), rank_str, fill='black', font=font)
        draw.text((lx, ly), rank_str, fill=color, font=font)

        est_z = c.get('estimated_depth_m', pos_3d[2] if pos_3d else None)
        angle_str = f"{angle_deg:.0f}°" if angle_deg is not None else "?"
        line2 = f"Z~{est_z:.2f}m  theta={angle_str}"
        info_y = ly + font.size + 2
        draw.text((lx + 1, info_y + 1), line2, fill='black', font=font_sm)
        draw.text((lx, info_y), line2, fill=light, font=font_sm)

        if pos_3d:
            line3 = f"({pos_3d[0]:.3f}, {pos_3d[1]:.3f}, {pos_3d[2]:.3f}) m"
            pos_y = info_y + font_sm.size + 2
            draw.text((lx + 1, pos_y + 1), line3, fill='black', font=font_sm)
            draw.text((lx, pos_y), line3, fill=light, font=font_sm)

    return img


# =============================================================================
# Build candidate dicts from parsed_6dof & draw
# =============================================================================

def build_6dof_candidate_dicts(parsed: list[Align6DoFResult]) -> list[dict]:
    """Convert parsed 6DOF results to the dict format expected by draw_web_ai_grasps."""
    candidates = []
    for r in parsed:
        c = {
            'rank': r.rank,
            'pixel_uv': list(r.pixel_uv) if r.pixel_uv else None,
            'gripper_angle_deg': r.angle_deg,
            'estimated_depth_m': r.estimated_depth_m,
        }
        if r.position_3d is not None:
            c['position_3d'] = r.position_3d.tolist()
        if r.closing_direction_3d is not None:
            c['closing_direction_3d'] = r.closing_direction_3d.tolist()
        if r.approach_direction_3d is not None:
            c['approach_direction_3d'] = r.approach_direction_3d.tolist()
        candidates.append(c)
    return candidates


candidates = build_6dof_candidate_dicts(parsed_6dof)
print(f'Drawing {len(candidates)} candidate(s) ...')

annotated = draw_web_ai_grasps(rgb_pil, candidates, K)

# Save
out_path = capture_dir / '6dof_grasp_viz.jpg'
annotated.save(out_path, quality=92)
print(f'Saved: {out_path}')

# Display inline
try:
    from IPython.display import display as ipy_display
    ipy_display(annotated)
except ImportError:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(14, 8))
    plt.imshow(annotated)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Part 9: Interactive 3D Grasp Visualization

An **interactive 3D plot** (Plotly) showing:
- The full scene as a colored **point cloud** (back-projected from depth + RGB)
- Each grasp candidate as a **gripper wireframe** with closing/approach arrows
- **XYZ axis indicators** per grasp: red=closing, green=lateral, blue=approach
- Comparison markers showing the VLM's estimated vs actual 3D position

Drag to rotate, scroll to zoom, hover for details. Reuses code from
`align_experiment.ipynb` Part 9.

In [ ]:
# =============================================================================
# 3-D interactive grasp visualization using Plotly
# Adapted from align_experiment.ipynb Part 9 to use parsed_6dof instead of
# a hardcoded WEB_AI_RESULT dict.
# =============================================================================

import plotly.graph_objects as go

# ── 1. Back-project the scene point cloud ──────────────────────────────────
depth_map_f64 = depth_map.astype(np.float64)
K_f64 = K.astype(np.float64)

# RGB as numpy for colour sampling
rgb_full = np.asarray(rgb_pil)  # (H, W, 3) uint8 RGB
valid_mask = np.isfinite(depth_map_f64) & (depth_map_f64 > 0.0)

print("Back-projecting depth map to point cloud ...")
pts_full = backproject_depth_with_mask(depth_map_f64, K_f64)  # (N, 3) float32
colors_full = rgb_full[valid_mask]                              # (N, 3) uint8
print(f"  Full point cloud: {pts_full.shape[0]:,} points")

# Subsample for smooth interactivity
MAX_PTS = 15000
if pts_full.shape[0] > MAX_PTS:
    idx = np.random.choice(pts_full.shape[0], MAX_PTS, replace=False)
    pts = pts_full[idx]
    cols = colors_full[idx]
    print(f"  Downsampled to {MAX_PTS:,} points for rendering")
else:
    pts = pts_full
    cols = colors_full

# ── 2. Gripper wireframe helper ────────────────────────────────────────────
def _gripper_wireframe(pose_4x4, width_m):
    """Return (N,3) world points + list of (start, end) edge index pairs."""
    w = width_m if (width_m is not None and width_m > 0) else 0.08
    R = pose_4x4[:3, :3]
    t = pose_4x4[:3, 3]
    local = np.array([
        [0.0,     0.0, 0.0],                                    # 0: palm center
        [0.0,     0.0, GRIPPER_DEPTH_METERS],                   # 1: palm base
        [-w / 2,  0.0, GRIPPER_DEPTH_METERS],                   # 2: left base
        [-w / 2,  0.0, GRIPPER_DEPTH_METERS + FINGER_LENGTH_METERS],  # 3: left tip
        [ w / 2,  0.0, GRIPPER_DEPTH_METERS],                   # 4: right base
        [ w / 2,  0.0, GRIPPER_DEPTH_METERS + FINGER_LENGTH_METERS],  # 5: right tip
    ], dtype=np.float32)
    world = (local @ R.T) + t.reshape(1, 3)
    edges = [(0, 1), (2, 3), (4, 5), (2, 4)]
    return world, edges


# ── 3. Build per-candidate data: actual 3D position + 4x4 pose ────────────
candidates_3d = []

for r in parsed_6dof:
    if r.pixel_uv is None:
        print(f"  [WARN] Candidate #{r.rank}: missing pixel_uv, skipping")
        continue

    u, v = r.pixel_uv

    # Sample actual depth at the VLM-chosen pixel
    try:
        z_actual = sample_depth_median(depth_map_f64, v, u, window=5)
    except ValueError:
        print(f"  [WARN] Candidate #{r.rank}: no valid depth at ({u}, {v}), skipping")
        continue

    # Deproject using actual intrinsics
    pos_actual = deproject_pixel(u, v, z_actual, K_f64)

    # Closing direction from VLM (fall back to angle_deg)
    close_dir = r.closing_direction_3d
    if close_dir is None:
        th = np.radians(r.angle_deg)
        close_dir = np.array([np.cos(th), np.sin(th), 0.0], dtype=np.float64)

    approach = np.array([0.0, 0.0, 1.0], dtype=np.float64)
    lateral = np.cross(approach, close_dir)
    lat_norm = np.linalg.norm(lateral)
    if lat_norm < 1e-9:
        lateral = np.array([1.0, 0.0, 0.0], dtype=np.float64)
    else:
        lateral = lateral / lat_norm

    # Build 4x4 pose matrix (camera frame)
    pose = np.eye(4, dtype=np.float64)
    pose[:3, 0] = close_dir
    pose[:3, 1] = lateral
    pose[:3, 2] = approach
    pose[:3, 3] = pos_actual

    ai_pos = r.position_3d

    dist_mm = 0.0
    if ai_pos is not None:
        dist_mm = np.linalg.norm(np.array(ai_pos) - pos_actual) * 1000

    print(f"  #{r.rank}: pixel=({u},{v})  Z_actual={z_actual:.3f}m  "
          f"pos={pos_actual.round(4)}  VLM_offset={dist_mm:.1f}mm")

    candidates_3d.append({
        'rank': r.rank,
        'pos_actual': pos_actual,
        'z_actual': z_actual,
        'pose': pose,
        'close_dir': close_dir,
        'ai_pos_3d': np.array(ai_pos, dtype=np.float64) if ai_pos is not None else None,
        'angle_deg': r.angle_deg,
        'estimated_width_m': r.estimated_width_m,
    })

if not candidates_3d:
    raise RuntimeError("No valid candidates for 3D visualization. Check pixel_uv values.")

print(f"\nReady to render {len(candidates_3d)} grasp(s) in 3D.")

# ── 4. Build Plotly figure ────────────────────────────────────────────────
fig = go.Figure()

# ---- Scene point cloud ----
color_strs = [f'rgb({r},{g},{b})' for r, g, b in cols]
fig.add_trace(go.Scatter3d(
    x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
    mode='markers',
    name='scene point cloud',
    marker={'size': 2, 'opacity': 0.6, 'color': color_strs},
    hoverinfo='skip',
))

# ---- Per-candidate artifacts ----
AXIS_LENGTH = 0.05  # metres

for i, c in enumerate(candidates_3d):
    rank = c['rank']
    color = _rank_color(i, len(candidates_3d))
    rgb_str = f'rgb({color[0]},{color[1]},{color[2]})'
    rgb_faint = f'rgba({color[0]},{color[1]},{color[2]},0.45)'

    pose = c['pose']
    center = c['pos_actual']
    rotation = pose[:3, :3]
    width_m = c.get('estimated_width_m', 0.05)

    # -- Gripper wireframe --
    wf_pts, wf_edges = _gripper_wireframe(pose, width_m)
    for s_idx, e_idx in wf_edges:
        seg = wf_pts[[s_idx, e_idx]]
        fig.add_trace(go.Scatter3d(
            x=seg[:, 0], y=seg[:, 1], z=seg[:, 2],
            mode='lines',
            line={'color': rgb_str, 'width': 6},
            hoverinfo='skip',
            showlegend=False,
        ))

    # -- XYZ axis arrows --
    AXIS_SPECS = [
        ('X closing',  0, 'rgb(255,90,90)'),
        ('Y lateral',  1, 'rgb(90,220,100)'),
        ('Z approach', 2, 'rgb(80,150,255)'),
    ]
    for ax_name, col_idx, ax_color in AXIS_SPECS:
        tip = center + AXIS_LENGTH * rotation[:, col_idx]
        fig.add_trace(go.Scatter3d(
            x=[center[0], tip[0]],
            y=[center[1], tip[1]],
            z=[center[2], tip[2]],
            mode='lines',
            name=ax_name if i == 0 else None,
            line={'color': ax_color, 'width': 5},
            hoverinfo='skip',
            showlegend=(i == 0),
        ))

    # -- Center marker with rank label --
    angle_str = f"{c.get('angle_deg', '?'):.0f}°"
    fig.add_trace(go.Scatter3d(
        x=[center[0]], y=[center[1]], z=[center[2]],
        mode='markers+text',
        name=f"#{rank}",
        text=[f"#{rank}"],
        textposition='top center',
        textfont={'size': 12, 'color': rgb_str},
        marker={'size': 8, 'color': rgb_str, 'line': {'color': 'black', 'width': 1}},
        hovertemplate=(
            f"<b>#{rank}</b><br>"
            f"pos=({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f}) m<br>"
            f"Z={c['z_actual']:.3f} m<br>"
            f"theta={angle_str}<br>"
            f"width~{width_m*1000:.0f} mm"
            f"<extra></extra>"
        ),
    ))

    # -- VLM's estimated position (if differs by > 1 cm) --
    ai_pos = c.get('ai_pos_3d')
    if ai_pos is not None:
        dist_m = np.linalg.norm(ai_pos - center)
        if dist_m > 0.01:
            fig.add_trace(go.Scatter3d(
                x=[ai_pos[0]], y=[ai_pos[1]], z=[ai_pos[2]],
                mode='markers',
                marker={'size': 5, 'color': rgb_str, 'symbol': 'diamond-open',
                        'line': {'width': 2}},
                name=f"VLM est #{rank}" if i == 0 else None,
                showlegend=(i == 0),
                hovertemplate=(
                    f"<b>VLM estimated #{rank}</b><br>"
                    f"offset from actual: {dist_m*100:.1f} cm<br>"
                    f"position=({ai_pos[0]:.3f}, {ai_pos[1]:.3f}, {ai_pos[2]:.3f})"
                    f"<extra></extra>"
                ),
            ))
            # Dashed connector
            fig.add_trace(go.Scatter3d(
                x=[ai_pos[0], center[0]],
                y=[ai_pos[1], center[1]],
                z=[ai_pos[2], center[2]],
                mode='lines',
                line={'color': rgb_faint, 'width': 1, 'dash': 'dot'},
                hoverinfo='skip',
                showlegend=False,
            ))

# ── 5. Layout & display ───────────────────────────────────────────────────
fig.update_layout(
    title=(
        f"<b>3D Grasp Visualization</b> — {task_spec}<br>"
        f"<sup>XYZ axes per grasp: "
        f"<span style='color:#ff5a5a'>red=closing</span>  "
        f"<span style='color:#5adc64'>green=lateral</span>  "
        f"<span style='color:#5096ff'>blue=approach (+Z)</span></sup>"
    ),
    margin={'l': 0, 'r': 0, 't': 80, 'b': 0},
    scene={
        'xaxis_title': 'X (m) -> right',
        'yaxis_title': 'Y (m) -> down',
        'zaxis_title': 'Z (m) -> into scene',
        'aspectmode': 'data',
    },
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': -0.15},
    hovermode='closest',
)

# Inline display
fig.show()

# Save as self-contained HTML
html_path = capture_dir / '6dof_grasp_viz_3d.html'
fig.write_html(str(html_path), include_plotlyjs=True, full_html=True)
print(f"\nSaved interactive 3D view: {html_path}")